In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import os
import json
import cv2
import shutil
import random


In [3]:
ROOT_PATH = "/content/drive/MyDrive/Leukemiadata"

SUBFOLDERS = [
    "H_100X_C1",
    "H_100X_C2",
    "L_100X_C1"
]

OUTPUT_BASE = "/content/wbc_dataset"
TRAIN_OUT = os.path.join(OUTPUT_BASE, "train")
TEST_OUT = os.path.join(OUTPUT_BASE, "test")


In [4]:
all_classes = set()

for folder in SUBFOLDERS:
    json_dir = os.path.join(ROOT_PATH, folder, "json_labels")
    for split in ["train.json", "test.json"]:
        with open(os.path.join(json_dir, split), "r") as f:
            data = json.load(f)
            for ann in data["annotations"]:
                all_classes.add(ann["category_name"])

classes = sorted(all_classes)
print("WBC classes found:", classes)


WBC classes found: ['abnormal promyelocyte', 'atypical lymphocyte', 'basophil', 'eosinophil', 'lymphoblast', 'lymphocyte', 'metamyelocyte', 'monoblast', 'monocyte', 'myeloblast', 'myelocyte', 'neutrophil', 'none', 'promonocyte']


In [5]:
if os.path.exists(OUTPUT_BASE):
    shutil.rmtree(OUTPUT_BASE)

for cls in classes:
    os.makedirs(os.path.join(TRAIN_OUT, cls), exist_ok=True)
    os.makedirs(os.path.join(TEST_OUT, cls), exist_ok=True)

print("Output folders created")


Output folders created


In [6]:
def process_split(folder, split):
    """
    folder: H_100x_C1, etc.
    split: 'train' or 'test'
    """
    base = os.path.join(ROOT_PATH, folder)

    image_dir = os.path.join(base, "Images", split)
    json_path = os.path.join(base, "json_labels", f"{split}.json")

    with open(json_path, "r") as f:
        data = json.load(f)

    image_id_to_name = {img["id"]: img["file_name"] for img in data["images"]}

    saved = 0

    for ann in data["annotations"]:
        img_name = image_id_to_name[ann["image_id"]]
        label = ann["category_name"]

        img_path = os.path.join(image_dir, img_name)
        image = cv2.imread(img_path)

        if image is None:
            continue

        x, y, w, h = map(int, ann["bbox"])
        cell = image[y:y+h, x:x+w]

        if cell.size == 0:
            continue

        cell = cv2.resize(cell, (224, 224))

        out_dir = TRAIN_OUT if split == "train" else TEST_OUT
        save_path = os.path.join(out_dir, label, f"{folder}_{split}_{saved}.jpg")

        cv2.imwrite(save_path, cell)
        saved += 1

    print(f"{folder} | {split}: {saved} cells saved")


In [7]:
for folder in SUBFOLDERS:
    process_split(folder, "train")
    process_split(folder, "test")

print("✅ All cell images extracted correctly")


H_100X_C1 | train: 7927 cells saved
H_100X_C1 | test: 2189 cells saved
H_100X_C2 | train: 8054 cells saved
H_100X_C2 | test: 2202 cells saved
L_100X_C1 | train: 0 cells saved
L_100X_C1 | test: 0 cells saved
✅ All cell images extracted correctly


In [8]:
for cls in classes:
    train_count = len(os.listdir(os.path.join(TRAIN_OUT, cls)))
    test_count = len(os.listdir(os.path.join(TEST_OUT, cls)))
    print(f"{cls}: Train={train_count}, Test={test_count}")


abnormal promyelocyte: Train=226, Test=319
atypical lymphocyte: Train=444, Test=394
basophil: Train=31, Test=6
eosinophil: Train=106, Test=64
lymphoblast: Train=3038, Test=384
lymphocyte: Train=462, Test=176
metamyelocyte: Train=150, Test=52
monoblast: Train=723, Test=240
monocyte: Train=198, Test=211
myeloblast: Train=3685, Test=556
myelocyte: Train=362, Test=243
neutrophil: Train=1565, Test=1007
none: Train=4345, Test=697
promonocyte: Train=646, Test=42


In [9]:
TRAIN_PATH = "/content/wbc_dataset/train"
TEST_PATH = "/content/wbc_dataset/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32


In [10]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_data = test_gen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_data.class_indices.keys())
print("Classes:", class_names)


Found 15981 images belonging to 14 classes.
Found 4391 images belonging to 14 classes.
Classes: ['abnormal promyelocyte', 'atypical lymphocyte', 'basophil', 'eosinophil', 'lymphoblast', 'lymphocyte', 'metamyelocyte', 'monoblast', 'monocyte', 'myeloblast', 'myelocyte', 'neutrophil', 'none', 'promonocyte']


In [11]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # IMPORTANT for beginners

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │         3,598 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,589,518 (9.88 MB)

 Trainable params: 331,534 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
EPOCHS = 15

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS
)


In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title("Loss")
plt.legend()

plt.show()


In [ ]:
y_true = test_data.classes
y_pred = np.argmax(model.predict(test_data), axis=1)


In [ ]:
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
model.save("/content/drive/MyDrive/wbc_cell_classifier.h5")
print("✅ Model saved to Google Drive")
